# 13 — equation-level proxy benchmark（30シナリオ）

## 背景
パラメータを説明するだけでは、制御モデルがどこまで要求wrenchを実現できるか分からない。
そこで、`legged_control` の中心式であるcentroidal dynamics、接触schedule、摩擦制約、
WBCのtorque limit、100 Hz計画を一つの小さな閉ループへまとめ、難易度別30条件を同じ指標で流す。
これは4秒の **equation-level pre-benchmark** であり、repository performanceではない。

## 重要な測定範囲
これは `external/legged_control` のROS node、OCS2 SQP、Pinocchio WBC、Gazebo A1を起動した
end-to-end benchmarkではない。この環境には `roscore`, `roslaunch`, `catkin` が無いためである。
ここで測るのは **上流と同じ物理契約を持つ教育用model-level benchmark**:

- centroidal合力・合moment
- stance/trot接触schedule
- 摩擦pyramid投影
- 33.5 N m torque proxy limit
- 状態noise、policy delay、外乱

したがって、結果は実機歩行性能ではなく、要求wrenchの実現性・tracking・計算時間の比較である。

## 目的
1. 30条件を同一コードで再実行する。
2. tracking、姿勢、高さ、wrench residual、torque飽和、計算時間を保存する。
3. 難しくなるほど、どの制約が先に支配的になるかを分析する。

## 結論の読み方
`pass` はこの縮約modelの閾値を満たした意味だけを持つ。上流repositoryの性能を断定するには、
ROS Noetic + OCS2 + Gazebo環境で同じscenario定義を移植し直す必要がある。


## 検証範囲に関する必須注記

このprojectでは **ROS2 portを作成・compile・実行していない**。したがってROS2 parityは
**NOT VERIFIED / FAIL-CLOSED** である。上流commit `a7f381c0367e98e31c01336e678eef47e304d40d` はROS1実装であり、
project所有MuJoCo adapterはOCS2のhorizon SQPを瞬時force plannerへ、
Pinocchio/qpOASES WBCをMuJoCo acceleration inverse dynamicsへ置換し、
元のestimator/hardware経路も持たない。保存済み30 scenario dataが示すのはadapter挙動だけで、
上流 `legged_control` の性能でもROS2移行の検証でもない。


In [ ]:
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`from pathlib import Path` の依存を明示して再現可能な実行環境を作る。
from pathlib import Path
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`import numpy as np` の依存を明示して再現可能な実行環境を作る。
import numpy as np
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`import matplotlib.pyplot as plt` の依存を明示して再現可能な実行環境を作る。
import matplotlib.pyplot as plt

# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`ROOT` を後続計算で使う明示的な中間量として設定する。 数式: `ROOT = Path.cwd()` の演算・変換をPythonで評価する。
ROOT = Path.cwd()
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`for candidate in [ROOT, *ROOT.parents]:` の反復範囲を固定して各sampleを処理する。 数式: `for candidate in [ROOT, *ROOT.parents]:` の演算・変換をPythonで評価する。
for candidate in [ROOT, *ROOT.parents]:
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`if (candidate / "pyproject.toml").exists():` の条件で安全側の実行分岐を選ぶ。 数式: `if (candidate / "pyproject.toml").exists():` の演算・変換をPythonで評価する。
    if (candidate / "pyproject.toml").exists():
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`ROOT` を後続計算で使う明示的な中間量として設定する。 数式: `ROOT = candidate` の演算・変換をPythonで評価する。
        ROOT = candidate
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`break` をこの章の処理順に沿って実行する。
        break

# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`np.set_printoptions(precision` を後続計算で使う明示的な中間量として設定する。 数式: `np.set_printoptions(precision=4, suppress=True)` の演算・変換をPythonで評価する。
np.set_printoptions(precision=4, suppress=True)
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})` の要素または終端を対応付ける。
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`print("repository:", ROOT)` の観測値を表示して判定根拠を残す。
print("repository:", ROOT)

# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`import json` の依存を明示して再現可能な実行環境を作る。
import json
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`import time` の依存を明示して再現可能な実行環境を作る。
import time
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`import pandas as pd` の依存を明示して再現可能な実行環境を作る。
import pandas as pd

# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`OUT` を後続計算で使う明示的な中間量として設定する。 数式: `OUT = ROOT / "notebook_legged" / "assets"` の演算・変換をPythonで評価する。
OUT = ROOT / "notebook_legged" / "assets"
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`OUT.mkdir(parents` を後続計算で使う明示的な中間量として設定する。 数式: `OUT.mkdir(parents=True, exist_ok=True)` の演算・変換をPythonで評価する。
OUT.mkdir(parents=True, exist_ok=True)


## データの流れ

```text
Scenario
  │ vx_ref, slope, mu, gait, noise, delay, disturbance
  ▼
reference + noisy measured state
  │
  ▼ 100 Hz
centroidal feedback
  │ desired wrench w*=[Fx,Fy,Fz,Mx,My,Mz]
  ▼
contact force allocation
  │ min ||A(contact)F - w*||² + λ||F||²
  ▼
friction/normal-force projection
  │ Fz>=0, |Fx|<=mu Fz, |Fy|<=mu Fz
  ▼
WBC torque proxy
  │ tau_i = J_i^T F_i, |tau|<=33.5 N m
  ▼
delayed plant integration
  │ m vdot = ΣF + mg + disturbance
  │ I ωdot = Σ(r_i×F_i) + disturbance moment
  └──────────── state ───────────→ feedback
```

上流との対応:

- desired wrench / centroidal model → `LeggedRobotDynamicsAD`
- contact schedule → `GaitSchedule`
- force constraint → NMPC `FrictionConeConstraint`
- force allocation → NMPCの現在入力 $u^*[0:12]$ の縮約
- torque proxy → WBCの $M\ddot q-J^TF-S^T\tau+nle=0$ の縮約


## 数式

controller:
\[
F_x^*=m k_v(v_x^{ref}-\hat v_x),\qquad
F_z^*=m[g+k_z(z^{ref}-\hat z)-d_z\hat v_z]
\]
\[
M_{x,y}^*=k_R(R^{ref}_{x,y}-\hat R_{x,y})-d_R\hat\omega_{x,y}
\]

active contactの力を積んだ $F$ に対し、
\[
\min_F \|AF-w^*\|_2^2+\lambda\|F\|_2^2
\]
を解いた後、各足を摩擦pyramidへ投影する。

plant:
\[
\dot v=\frac{1}{m}\left(\sum_iF_i+F_{ext}\right)+g,\qquad
\dot\omega=I^{-1}\left(\sum_i r_i\times F_i+M_{ext}\right).
\]

C++の完全なSQP/WBCを置き換えるものではないが、入力・制約・残差の意味は同じである。


In [ ]:
# --- Block A: scenario contract ---
# 内容: 30条件で変える物理量を一つの辞書形式へ揃える。
# 意図: 難易度ごとに別コードを書かず、controllerの性能差だけを比較する。
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`BASE` を後続計算で使う明示的な中間量として設定する。 数式: `BASE = dict(` の演算・変換をPythonで評価する。
BASE = dict(
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`duration` を後続計算で使う明示的な中間量として設定する。 数式: `duration=4.0, dt=0.01, gait="stance", gait_hz=1.35,` の演算・変換をPythonで評価する。
    duration=4.0, dt=0.01, gait="stance", gait_hz=1.35,
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`vx_ref` を後続計算で使う明示的な中間量として設定する。 数式: `vx_ref=0.0, slope_deg=0.0, mu=0.40, mass_scale=1.0,` の演算・変換をPythonで評価する。
    vx_ref=0.0, slope_deg=0.0, mu=0.40, mass_scale=1.0,
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`noise_pos` を後続計算で使う明示的な中間量として設定する。 数式: `noise_pos=0.0, noise_vel=0.0, delay_ms=0.0,` の演算・変換をPythonで評価する。
    noise_pos=0.0, noise_vel=0.0, delay_ms=0.0,
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`force_x` を後続計算で使う明示的な中間量として設定する。 数式: `force_x=0.0, force_y=0.0, moment_roll=0.0,` の演算・変換をPythonで評価する。
    force_x=0.0, force_y=0.0, moment_roll=0.0,
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `)` の要素または終端を対応付ける。
)

# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario` の責務を独立関数として定義する。 数式: `def scenario(name, level, **changes):` の演算・変換をPythonで評価する。
def scenario(name, level, **changes):
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`cfg` を後続計算で使う明示的な中間量として設定する。 数式: `cfg = BASE | changes` の演算・変換をPythonで評価する。
    cfg = BASE | changes
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`return {"name": name, "level": level, **cfg}` の値を次の制御境界へ返す。 数式: `return {"name": name, "level": level, **cfg}` の演算・変換をPythonで評価する。
    return {"name": name, "level": level, **cfg}

# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`SCENARIOS` を後続計算で使う明示的な中間量として設定する。 数式: `SCENARIOS = [` の演算・変換をPythonで評価する。
SCENARIOS = [
    # 簡易: 四脚接地を中心に、単独の小変更だけを加える。
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `scenario("E01_static_nominal", "easy"),` の要素または終端を対応付ける。
    scenario("E01_static_nominal", "easy"),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("E02_static_mu050", "easy", mu` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("E02_static_mu050", "easy", mu=0.50),` の演算・変換をPythonで評価する。
    scenario("E02_static_mu050", "easy", mu=0.50),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("E03_static_mu030", "easy", mu` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("E03_static_mu030", "easy", mu=0.30),` の演算・変換をPythonで評価する。
    scenario("E03_static_mu030", "easy", mu=0.30),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("E04_forward_010", "easy", vx_ref` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("E04_forward_010", "easy", vx_ref=0.10),` の演算・変換をPythonで評価する。
    scenario("E04_forward_010", "easy", vx_ref=0.10),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("E05_forward_020", "easy", vx_ref` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("E05_forward_020", "easy", vx_ref=0.20),` の演算・変換をPythonで評価する。
    scenario("E05_forward_020", "easy", vx_ref=0.20),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("E06_slope_plus3", "easy", slope_deg` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("E06_slope_plus3", "easy", slope_deg=3.0),` の演算・変換をPythonで評価する。
    scenario("E06_slope_plus3", "easy", slope_deg=3.0),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("E07_slope_minus3", "easy", slope_deg` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("E07_slope_minus3", "easy", slope_deg=-3.0),` の演算・変換をPythonで評価する。
    scenario("E07_slope_minus3", "easy", slope_deg=-3.0),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("E08_mass_plus05", "easy", mass_scale` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("E08_mass_plus05", "easy", mass_scale=1.05),` の演算・変換をPythonで評価する。
    scenario("E08_mass_plus05", "easy", mass_scale=1.05),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("E09_noise_small", "easy", noise_pos` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("E09_noise_small", "easy", noise_pos=0.002, noise_vel=0.01),` の演算・変換をPythonで評価する。
    scenario("E09_noise_small", "easy", noise_pos=0.002, noise_vel=0.01),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("E10_push_5N", "easy", force_x` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("E10_push_5N", "easy", force_x=5.0),` の演算・変換をPythonで評価する。
    scenario("E10_push_5N", "easy", force_x=5.0),

    # 普通: trot、速度、斜面、noise、delayを現実的範囲で一つまたは二つ組み合わせる。
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("N01_trot_static", "normal", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("N01_trot_static", "normal", gait="trot"),` の演算・変換をPythonで評価する。
    scenario("N01_trot_static", "normal", gait="trot"),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("N02_trot_v020", "normal", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("N02_trot_v020", "normal", gait="trot", vx_ref=0.20),` の演算・変換をPythonで評価する。
    scenario("N02_trot_v020", "normal", gait="trot", vx_ref=0.20),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("N03_trot_v040", "normal", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("N03_trot_v040", "normal", gait="trot", vx_ref=0.40),` の演算・変換をPythonで評価する。
    scenario("N03_trot_v040", "normal", gait="trot", vx_ref=0.40),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("N04_trot_slope5", "normal", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("N04_trot_slope5", "normal", gait="trot", slope_deg=5.0),` の演算・変換をPythonで評価する。
    scenario("N04_trot_slope5", "normal", gait="trot", slope_deg=5.0),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("N05_trot_mu030", "normal", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("N05_trot_mu030", "normal", gait="trot", mu=0.30),` の演算・変換をPythonで評価する。
    scenario("N05_trot_mu030", "normal", gait="trot", mu=0.30),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("N06_trot_mass115", "normal", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("N06_trot_mass115", "normal", gait="trot", mass_scale=1.15),` の演算・変換をPythonで評価する。
    scenario("N06_trot_mass115", "normal", gait="trot", mass_scale=1.15),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("N07_trot_noise", "normal", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("N07_trot_noise", "normal", gait="trot", noise_pos=0.005, noise…` の演算・変換をPythonで評価する。
    scenario("N07_trot_noise", "normal", gait="trot", noise_pos=0.005, noise_vel=0.03),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("N08_trot_delay10", "normal", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("N08_trot_delay10", "normal", gait="trot", delay_ms=10.0),` の演算・変換をPythonで評価する。
    scenario("N08_trot_delay10", "normal", gait="trot", delay_ms=10.0),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("N09_trot_push15", "normal", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("N09_trot_push15", "normal", gait="trot", force_y=15.0),` の演算・変換をPythonで評価する。
    scenario("N09_trot_push15", "normal", gait="trot", force_y=15.0),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("N10_v040_slope5", "normal", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("N10_v040_slope5", "normal", gait="trot", vx_ref=0.40, slope_de…` の演算・変換をPythonで評価する。
    scenario("N10_v040_slope5", "normal", gait="trot", vx_ref=0.40, slope_deg=5.0),

    # 高難度: 低摩擦・大速度・大外乱・大delayを組み合わせ、制約支配を観察する。
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("H01_v080", "hard", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("H01_v080", "hard", gait="trot", vx_ref=0.80),` の演算・変換をPythonで評価する。
    scenario("H01_v080", "hard", gait="trot", vx_ref=0.80),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("H02_mu018", "hard", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("H02_mu018", "hard", gait="trot", vx_ref=0.40, mu=0.18),` の演算・変換をPythonで評価する。
    scenario("H02_mu018", "hard", gait="trot", vx_ref=0.40, mu=0.18),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("H03_slope12", "hard", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("H03_slope12", "hard", gait="trot", vx_ref=0.30, slope_deg=12.0…` の演算・変換をPythonで評価する。
    scenario("H03_slope12", "hard", gait="trot", vx_ref=0.30, slope_deg=12.0),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("H04_mass140", "hard", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("H04_mass140", "hard", gait="trot", mass_scale=1.40),` の演算・変換をPythonで評価する。
    scenario("H04_mass140", "hard", gait="trot", mass_scale=1.40),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("H05_delay30", "hard", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("H05_delay30", "hard", gait="trot", vx_ref=0.50, delay_ms=30.0)…` の演算・変換をPythonで評価する。
    scenario("H05_delay30", "hard", gait="trot", vx_ref=0.50, delay_ms=30.0),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("H06_noise_large", "hard", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("H06_noise_large", "hard", gait="trot", vx_ref=0.40, noise_pos=…` の演算・変換をPythonで評価する。
    scenario("H06_noise_large", "hard", gait="trot", vx_ref=0.40, noise_pos=0.02, noise_vel=0.12),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("H07_push40", "hard", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("H07_push40", "hard", gait="trot", vx_ref=0.30, force_y=40.0),` の演算・変換をPythonで評価する。
    scenario("H07_push40", "hard", gait="trot", vx_ref=0.30, force_y=40.0),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("H08_roll12Nm", "hard", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("H08_roll12Nm", "hard", gait="trot", moment_roll=12.0),` の演算・変換をPythonで評価する。
    scenario("H08_roll12Nm", "hard", gait="trot", moment_roll=12.0),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("H09_combo_lowmu_slope", "hard", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("H09_combo_lowmu_slope", "hard", gait="trot", vx_ref=0.60, mu=0…` の演算・変換をPythonで評価する。
    scenario("H09_combo_lowmu_slope", "hard", gait="trot", vx_ref=0.60, mu=0.20, slope_deg=10.0),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenario("H10_combo_all", "hard", gait` を後続計算で使う明示的な中間量として設定する。 数式: `scenario("H10_combo_all", "hard", gait="trot", vx_ref=0.70, mu=0.18, slo…` の演算・変換をPythonで評価する。
    scenario("H10_combo_all", "hard", gait="trot", vx_ref=0.70, mu=0.18, slope_deg=12.0,
             # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`mass_scale` を後続計算で使う明示的な中間量として設定する。 数式: `mass_scale=1.25, delay_ms=30.0, noise_pos=0.02, noise_vel=0.12,` の演算・変換をPythonで評価する。
             mass_scale=1.25, delay_ms=30.0, noise_pos=0.02, noise_vel=0.12,
             # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`force_y` を後続計算で使う明示的な中間量として設定する。 数式: `force_y=30.0, moment_roll=8.0),` の演算・変換をPythonで評価する。
             force_y=30.0, moment_roll=8.0),
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `]` の要素または終端を対応付ける。
]

# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`assert len(SCENARIOS) == 30` を不変条件として即時検査する。 数式: `assert len(SCENARIOS) == 30` の演算・変換をPythonで評価する。
assert len(SCENARIOS) == 30
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `pd.DataFrame(SCENARIOS).groupby("level").size()` の要素または終端を対応付ける。
pd.DataFrame(SCENARIOS).groupby("level").size()


In [ ]:
# --- Block B: gait / contact schedule ---
# 数式: trotは対角2脚が半周期ごとに交代する。
# C++対応: GaitScheduleが返すcontactFlags(t)。NMPCがmodeを最適化するのではない。
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`LEG_NAMES` を後続計算で使う明示的な中間量として設定する。 数式: `LEG_NAMES = ("LF", "RF", "LH", "RH")` の演算・変換をPythonで評価する。
LEG_NAMES = ("LF", "RF", "LH", "RH")
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`FOOT_POS` を後続計算で使う明示的な中間量として設定する。 数式: `FOOT_POS = np.array([` の演算・変換をPythonで評価する。
FOOT_POS = np.array([
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `[ 0.25, 0.15, -0.30],` の要素または終端を対応付ける。 数式: `[ 0.25, 0.15, -0.30],` の演算・変換をPythonで評価する。
    [ 0.25,  0.15, -0.30],
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `[ 0.25, -0.15, -0.30],` の要素または終端を対応付ける。 数式: `[ 0.25, -0.15, -0.30],` の演算・変換をPythonで評価する。
    [ 0.25, -0.15, -0.30],
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `[-0.25, 0.15, -0.30],` の要素または終端を対応付ける。 数式: `[-0.25, 0.15, -0.30],` の演算・変換をPythonで評価する。
    [-0.25,  0.15, -0.30],
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `[-0.25, -0.15, -0.30],` の要素または終端を対応付ける。 数式: `[-0.25, -0.15, -0.30],` の演算・変換をPythonで評価する。
    [-0.25, -0.15, -0.30],
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `])` の要素または終端を対応付ける。
])

# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`contact_flags` の責務を独立関数として定義する。
def contact_flags(t, gait, gait_hz):
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`if gait == "stance":` の条件で安全側の実行分岐を選ぶ。 数式: `if gait == "stance":` の演算・変換をPythonで評価する。
    if gait == "stance":
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`return np.ones(4, dtype=bool)` の値を次の制御境界へ返す。 数式: `return np.ones(4, dtype=bool)` の演算・変換をPythonで評価する。
        return np.ones(4, dtype=bool)
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`phase` を後続計算で使う明示的な中間量として設定する。 数式: `phase = (t * gait_hz) % 1.0` の演算・変換をPythonで評価する。
    phase = (t * gait_hz) % 1.0
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`return np.array([1, 0, 0, 1], dtype=bool) if phase < 0.5 else np.array([…` の値を次の制御境界へ返す。 数式: `return np.array([1, 0, 0, 1], dtype=bool) if phase < 0.5 else np.array([…` の演算・変換をPythonで評価する。
    return np.array([1, 0, 0, 1], dtype=bool) if phase < 0.5 else np.array([0, 1, 1, 0], dtype=bool)

# --- Block C: wrench matrix A ---
# 各足力Fiは合力へI3、CoM momentへskew(ri)で寄与する: [ΣFi; Σri×Fi] = A F。
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`skew` の責務を独立関数として定義する。
def skew(r):
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`x, y, z` を後続計算で使う明示的な中間量として設定する。 数式: `x, y, z = r` の演算・変換をPythonで評価する。
    x, y, z = r
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`return np.array([[0, -z, y], [z, 0, -x], [-y, x, 0]])` の値を次の制御境界へ返す。 数式: `return np.array([[0, -z, y], [z, 0, -x], [-y, x, 0]])` の演算・変換をPythonで評価する。
    return np.array([[0, -z, y], [z, 0, -x], [-y, x, 0]])

# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`wrench_matrix` の責務を独立関数として定義する。
def wrench_matrix(active):
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`blocks_force` を後続計算で使う明示的な中間量として設定する。 数式: `blocks_force = [np.eye(3) for _ in active]` の演算・変換をPythonで評価する。
    blocks_force = [np.eye(3) for _ in active]
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`blocks_moment` を後続計算で使う明示的な中間量として設定する。 数式: `blocks_moment = [skew(FOOT_POS[i]) for i in active]` の演算・変換をPythonで評価する。
    blocks_moment = [skew(FOOT_POS[i]) for i in active]
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`return np.vstack([np.hstack(blocks_force), np.hstack(blocks_moment)])` の値を次の制御境界へ返す。 数式: `return np.vstack([np.hstack(blocks_force), np.hstack(blocks_moment)])` の演算・変換をPythonで評価する。
    return np.vstack([np.hstack(blocks_force), np.hstack(blocks_moment)])

# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`print("all stance A:", wrench_matrix(np.arange(4)).shape)` の観測値を表示して判定根拠を残す。 数式: `print("all stance A:", wrench_matrix(np.arange(4)).shape)` の演算・変換をPythonで評価する。
print("all stance A:", wrench_matrix(np.arange(4)).shape)
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`print("trot A:", wrench_matrix(np.array([0, 3])).shape)` の観測値を表示して判定根拠を残す。 数式: `print("trot A:", wrench_matrix(np.array([0, 3])).shape)` の演算・変換をPythonで評価する。
print("trot A:", wrench_matrix(np.array([0, 3])).shape)


In [ ]:
# --- Block D: force allocator + friction projection ---
# 数式: min ||AF-w||² + ridge||F||²。
# コメント: pseudoinverseでunconstrained解を得てから、各足をWBC型摩擦pyramidへ投影する。
# 注意: 上流NMPCのsoft円錐とqpOASESの厳密QPを再現するsolverではない。
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`project_foot_force` の責務を独立関数として定義する。
def project_foot_force(f, mu, fz_max):
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`fx, fy, fz` を後続計算で使う明示的な中間量として設定する。 数式: `fx, fy, fz = f` の演算・変換をPythonで評価する。
    fx, fy, fz = f
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`fz` を後続計算で使う明示的な中間量として設定する。 数式: `fz = np.clip(fz, 0.0, fz_max)` の演算・変換をPythonで評価する。
    fz = np.clip(fz, 0.0, fz_max)
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`fx` を後続計算で使う明示的な中間量として設定する。 数式: `fx = np.clip(fx, -mu*fz, mu*fz)` の演算・変換をPythonで評価する。
    fx = np.clip(fx, -mu*fz, mu*fz)
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`fy` を後続計算で使う明示的な中間量として設定する。 数式: `fy = np.clip(fy, -mu*fz, mu*fz)` の演算・変換をPythonで評価する。
    fy = np.clip(fy, -mu*fz, mu*fz)
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`return np.array([fx, fy, fz])` の値を次の制御境界へ返す。 数式: `return np.array([fx, fy, fz])` の演算・変換をPythonで評価する。
    return np.array([fx, fy, fz])

# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`allocate_forces` の責務を独立関数として定義する。
def allocate_forces(wrench_des, contact, mu, mass):
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`active` を後続計算で使う明示的な中間量として設定する。 数式: `active = np.flatnonzero(contact)` の演算・変換をPythonで評価する。
    active = np.flatnonzero(contact)
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`A` を後続計算で使う明示的な中間量として設定する。 数式: `A = wrench_matrix(active)` の演算・変換をPythonで評価する。
    A = wrench_matrix(active)
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`ridge` を後続計算で使う明示的な中間量として設定する。 数式: `ridge = 1e-5` の演算・変換をPythonで評価する。
    ridge = 1e-5
    # 正規方程式ではなくlstsqで解き、rank不足のtrotでも最小norm解を得る。
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`A_aug` を後続計算で使う明示的な中間量として設定する。 数式: `A_aug = np.vstack([A, np.sqrt(ridge)*np.eye(3*len(active))])` の演算・変換をPythonで評価する。
    A_aug = np.vstack([A, np.sqrt(ridge)*np.eye(3*len(active))])
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`b_aug` を後続計算で使う明示的な中間量として設定する。 数式: `b_aug = np.r_[wrench_des, np.zeros(3*len(active))]` の演算・変換をPythonで評価する。
    b_aug = np.r_[wrench_des, np.zeros(3*len(active))]
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`f_active` を後続計算で使う明示的な中間量として設定する。 数式: `f_active = np.linalg.lstsq(A_aug, b_aug, rcond=None)[0].reshape(-1, 3)` の演算・変換をPythonで評価する。
    f_active = np.linalg.lstsq(A_aug, b_aug, rcond=None)[0].reshape(-1, 3)
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`fz_max` を後続計算で使う明示的な中間量として設定する。 数式: `fz_max = 3.0 * mass * 9.81 / max(len(active), 1)` の演算・変換をPythonで評価する。
    fz_max = 3.0 * mass * 9.81 / max(len(active), 1)
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`f_active` を後続計算で使う明示的な中間量として設定する。 数式: `f_active = np.array([project_foot_force(f, mu, fz_max) for f in f_active…` の演算・変換をPythonで評価する。
    f_active = np.array([project_foot_force(f, mu, fz_max) for f in f_active])
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`forces` を後続計算で使う明示的な中間量として設定する。 数式: `forces = np.zeros((4, 3))` の演算・変換をPythonで評価する。
    forces = np.zeros((4, 3))
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`forces[active]` を後続計算で使う明示的な中間量として設定する。 数式: `forces[active] = f_active` の演算・変換をPythonで評価する。
    forces[active] = f_active
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`realized` を後続計算で使う明示的な中間量として設定する。 数式: `realized = A @ f_active.ravel()` の演算・変換をPythonで評価する。
    realized = A @ f_active.ravel()
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`residual` を後続計算で使う明示的な中間量として設定する。 数式: `residual = np.linalg.norm(realized - wrench_des) / max(np.linalg.norm(wr…` の演算・変換をPythonで評価する。
    residual = np.linalg.norm(realized - wrench_des) / max(np.linalg.norm(wrench_des), 1.0)
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`margin` を後続計算で使う明示的な中間量として設定する。 数式: `margin = np.min(mu*f_active[:,2] - np.max(np.abs(f_active[:,:2]), axis=1…` の演算・変換をPythonで評価する。
    margin = np.min(mu*f_active[:,2] - np.max(np.abs(f_active[:,:2]), axis=1))
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`return forces, realized, residual, margin` の値を次の制御境界へ返す。 数式: `return forces, realized, residual, margin` の演算・変換をPythonで評価する。
    return forces, realized, residual, margin

# --- Block E: WBC torque proxy ---
# 数式: tau≈J^T F。A1の脚長に近いlever armで3関節torqueを近似する。
# 上流WBCは42変数QPなので、このproxyはtorque飽和傾向だけを見る。
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`J_PROXY` を後続計算で使う明示的な中間量として設定する。 数式: `J_PROXY = np.array([[0.08, 0.00, 0.00],` の演算・変換をPythonで評価する。
J_PROXY = np.array([[0.08, 0.00, 0.00],
                    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `[0.00, 0.20, 0.10],` の要素または終端を対応付ける。
                    [0.00, 0.20, 0.10],
                    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `[0.00, 0.10, 0.20]])` の要素または終端を対応付ける。
                    [0.00, 0.10, 0.20]])
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`TAU_LIMIT` を後続計算で使う明示的な中間量として設定する。 数式: `TAU_LIMIT = 33.5` の演算・変換をPythonで評価する。
TAU_LIMIT = 33.5

# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`torque_proxy` の責務を独立関数として定義する。
def torque_proxy(forces):
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`tau` を後続計算で使う明示的な中間量として設定する。 数式: `tau = np.array([J_PROXY.T @ f for f in forces])` の演算・変換をPythonで評価する。
    tau = np.array([J_PROXY.T @ f for f in forces])
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`saturation` を後続計算で使う明示的な中間量として設定する。 数式: `saturation = np.mean(np.abs(tau) >= TAU_LIMIT)` の演算・変換をPythonで評価する。
    saturation = np.mean(np.abs(tau) >= TAU_LIMIT)
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`return np.clip(tau, -TAU_LIMIT, TAU_LIMIT), saturation` の値を次の制御境界へ返す。 数式: `return np.clip(tau, -TAU_LIMIT, TAU_LIMIT), saturation` の演算・変換をPythonで評価する。
    return np.clip(tau, -TAU_LIMIT, TAU_LIMIT), saturation


In [ ]:
# --- Block F: closed-loop simulation ---
# 処理順: noisy state→100 Hz controller→force allocation→delay→plant integration。
# 出力: tracking、姿勢、高さ、制約、torque、計算時間をscenarioごとに集約する。
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`run_scenario` の責務を独立関数として定義する。 数式: `def run_scenario(cfg, seed=42):` の演算・変換をPythonで評価する。
def run_scenario(cfg, seed=42):
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`rng` を後続計算で使う明示的な中間量として設定する。 数式: `rng = np.random.default_rng(seed)` の演算・変換をPythonで評価する。
    rng = np.random.default_rng(seed)
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`dt` を後続計算で使う明示的な中間量として設定する。 数式: `dt = cfg["dt"]` の演算・変換をPythonで評価する。
    dt = cfg["dt"]
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`steps` を後続計算で使う明示的な中間量として設定する。 数式: `steps = int(cfg["duration"] / dt)` の演算・変換をPythonで評価する。
    steps = int(cfg["duration"] / dt)
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`mass` を後続計算で使う明示的な中間量として設定する。 数式: `mass = 12.5 * cfg["mass_scale"]` の演算・変換をPythonで評価する。
    mass = 12.5 * cfg["mass_scale"]
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`inertia` を後続計算で使う明示的な中間量として設定する。 数式: `inertia = np.array([0.24, 0.55, 0.65]) * cfg["mass_scale"]` の演算・変換をPythonで評価する。
    inertia = np.array([0.24, 0.55, 0.65]) * cfg["mass_scale"]
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`slope` を後続計算で使う明示的な中間量として設定する。 数式: `slope = np.deg2rad(cfg["slope_deg"])` の演算・変換をPythonで評価する。
    slope = np.deg2rad(cfg["slope_deg"])
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`delay_steps` を後続計算で使う明示的な中間量として設定する。 数式: `delay_steps = int(round(cfg["delay_ms"] / 1000 / dt))` の演算・変換をPythonで評価する。
    delay_steps = int(round(cfg["delay_ms"] / 1000 / dt))

    # state = [z, vx, vy, vz, roll, pitch, wx, wy]
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`state` を後続計算で使う明示的な中間量として設定する。 数式: `state = np.array([0.30, 0.0, 0.0, 0.0, 0.0, slope, 0.0, 0.0])` の演算・変換をPythonで評価する。
    state = np.array([0.30, 0.0, 0.0, 0.0, 0.0, slope, 0.0, 0.0])
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`force_queue` を後続計算で使う明示的な中間量として設定する。 数式: `force_queue = [np.zeros((4, 3)) for _ in range(delay_steps + 1)]` の演算・変換をPythonで評価する。
    force_queue = [np.zeros((4, 3)) for _ in range(delay_steps + 1)]
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`command_forces` を後続計算で使う明示的な中間量として設定する。 数式: `command_forces = np.zeros((4, 3))` の演算・変換をPythonで評価する。
    command_forces = np.zeros((4, 3))
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`controller_times, logs` を後続計算で使う明示的な中間量として設定する。 数式: `controller_times, logs = [], []` の演算・変換をPythonで評価する。
    controller_times, logs = [], []
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`residuals, margins, tau_sats` を後続計算で使う明示的な中間量として設定する。 数式: `residuals, margins, tau_sats = [], [], []` の演算・変換をPythonで評価する。
    residuals, margins, tau_sats = [], [], []

    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`for k in range(steps):` の反復範囲を固定して各sampleを処理する。
    for k in range(steps):
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`t` を後続計算で使う明示的な中間量として設定する。 数式: `t = k * dt` の演算・変換をPythonで評価する。
        t = k * dt
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`contact` を後続計算で使う明示的な中間量として設定する。 数式: `contact = contact_flags(t, cfg["gait"], cfg["gait_hz"])` の演算・変換をPythonで評価する。
        contact = contact_flags(t, cfg["gait"], cfg["gait_hz"])
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`measured` を後続計算で使う明示的な中間量として設定する。 数式: `measured = state.copy()` の演算・変換をPythonで評価する。
        measured = state.copy()
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`measured[0] +` を後続計算で使う明示的な中間量として設定する。 数式: `measured[0] += rng.normal(0, cfg["noise_pos"])` の演算・変換をPythonで評価する。
        measured[0] += rng.normal(0, cfg["noise_pos"])
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`measured[1:4] +` を後続計算で使う明示的な中間量として設定する。 数式: `measured[1:4] += rng.normal(0, cfg["noise_vel"], 3)` の演算・変換をPythonで評価する。
        measured[1:4] += rng.normal(0, cfg["noise_vel"], 3)

        # 上流と同じ100 Hzで計画を更新し、間のplant stepでは直前commandを使う。
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`if k % max(1, int(round(0.01/dt))) == 0:` の条件で安全側の実行分岐を選ぶ。 数式: `if k % max(1, int(round(0.01/dt))) == 0:` の演算・変換をPythonで評価する。
        if k % max(1, int(round(0.01/dt))) == 0:
            # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`tic` を後続計算で使う明示的な中間量として設定する。 数式: `tic = time.perf_counter()` の演算・変換をPythonで評価する。
            tic = time.perf_counter()
            # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`z, vx, vy, vz, roll, pitch, wx, wy` を後続計算で使う明示的な中間量として設定する。 数式: `z, vx, vy, vz, roll, pitch, wx, wy = measured` の演算・変換をPythonで評価する。
            z, vx, vy, vz, roll, pitch, wx, wy = measured
            # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`ax_des` を後続計算で使う明示的な中間量として設定する。 数式: `ax_des = 2.0 * (cfg["vx_ref"] - vx)` の演算・変換をPythonで評価する。
            ax_des = 2.0 * (cfg["vx_ref"] - vx)
            # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`ay_des` を後続計算で使う明示的な中間量として設定する。 数式: `ay_des = -2.0 * vy` の演算・変換をPythonで評価する。
            ay_des = -2.0 * vy
            # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`az_des` を後続計算で使う明示的な中間量として設定する。 数式: `az_des = 35.0 * (0.30 - z) - 9.0 * vz` の演算・変換をPythonで評価する。
            az_des = 35.0 * (0.30 - z) - 9.0 * vz
            # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`mx_des` を後続計算で使う明示的な中間量として設定する。 数式: `mx_des = 18.0 * (0.0 - roll) - 3.0 * wx` の演算・変換をPythonで評価する。
            mx_des = 18.0 * (0.0 - roll) - 3.0 * wx
            # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`my_des` を後続計算で使う明示的な中間量として設定する。 数式: `my_des = 18.0 * (slope - pitch) - 3.0 * wy` の演算・変換をPythonで評価する。
            my_des = 18.0 * (slope - pitch) - 3.0 * wy
            # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`wrench_des` を後続計算で使う明示的な中間量として設定する。 数式: `wrench_des = np.array([` の演算・変換をPythonで評価する。
            wrench_des = np.array([
                # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `mass*ax_des, mass*ay_des, mass*(9.81 + az_des),` の要素または終端を対応付ける。 数式: `mass*ax_des, mass*ay_des, mass*(9.81 + az_des),` の演算・変換をPythonで評価する。
                mass*ax_des, mass*ay_des, mass*(9.81 + az_des),
                # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `mx_des, my_des, 0.0,` の要素または終端を対応付ける。
                mx_des, my_des, 0.0,
            # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `])` の要素または終端を対応付ける。
            ])
            # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`command_forces, realized, residual, margin` を後続計算で使う明示的な中間量として設定する。 数式: `command_forces, realized, residual, margin = allocate_forces(` の演算・変換をPythonで評価する。
            command_forces, realized, residual, margin = allocate_forces(
                # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`wrench_des, contact, cfg["mu"], mass` をこの章の処理順に沿って実行する。
                wrench_des, contact, cfg["mu"], mass
            # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `)` の要素または終端を対応付ける。
            )
            # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`_, tau_sat` を後続計算で使う明示的な中間量として設定する。 数式: `_, tau_sat = torque_proxy(command_forces)` の演算・変換をPythonで評価する。
            _, tau_sat = torque_proxy(command_forces)
            # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `controller_times.append(time.perf_counter() - tic)` の要素または終端を対応付ける。 数式: `controller_times.append(time.perf_counter() - tic)` の演算・変換をPythonで評価する。
            controller_times.append(time.perf_counter() - tic)
            # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `residuals.append(residual); margins.append(margin); tau_sats.append(tau_…` の要素または終端を対応付ける。
            residuals.append(residual); margins.append(margin); tau_sats.append(tau_sat)

        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `force_queue.append(command_forces.copy())` の要素または終端を対応付ける。
        force_queue.append(command_forces.copy())
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`applied` を後続計算で使う明示的な中間量として設定する。 数式: `applied = force_queue.pop(0)` の演算・変換をPythonで評価する。
        applied = force_queue.pop(0)
        # slope座標の縮約: 重力を斜面接線(-x)と法線(-z)へ分解する。
        # 完全な3D接触frame回転ではないが、slopeを無意味なラベルにしない。
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`gravity_on_slope` を後続計算で使う明示的な中間量として設定する。 数式: `gravity_on_slope = np.array([-mass*9.81*np.sin(slope), 0.0, -mass*9.81*n…` の演算・変換をPythonで評価する。
        gravity_on_slope = np.array([-mass*9.81*np.sin(slope), 0.0, -mass*9.81*np.cos(slope)])
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`net_force` を後続計算で使う明示的な中間量として設定する。 数式: `net_force = applied.sum(axis=0) + gravity_on_slope + np.array([cfg["forc…` の演算・変換をPythonで評価する。
        net_force = applied.sum(axis=0) + gravity_on_slope + np.array([cfg["force_x"], cfg["force_y"], 0.0])
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`net_moment` を後続計算で使う明示的な中間量として設定する。 数式: `net_moment = np.cross(FOOT_POS, applied).sum(axis=0) + np.array([cfg["mo…` の演算・変換をPythonで評価する。
        net_moment = np.cross(FOOT_POS, applied).sum(axis=0) + np.array([cfg["moment_roll"], 0.0, 0.0])

        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`acc` を後続計算で使う明示的な中間量として設定する。 数式: `acc = net_force / mass` の演算・変換をPythonで評価する。
        acc = net_force / mass
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`alpha` を後続計算で使う明示的な中間量として設定する。 数式: `alpha = net_moment[:2] / inertia[:2]` の演算・変換をPythonで評価する。
        alpha = net_moment[:2] / inertia[:2]
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`state[1:4] +` を後続計算で使う明示的な中間量として設定する。 数式: `state[1:4] += dt * acc` の演算・変換をPythonで評価する。
        state[1:4] += dt * acc
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`state[0] +` を後続計算で使う明示的な中間量として設定する。 数式: `state[0] += dt * state[3]` の演算・変換をPythonで評価する。
        state[0] += dt * state[3]
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`state[6:8] +` を後続計算で使う明示的な中間量として設定する。 数式: `state[6:8] += dt * alpha` の演算・変換をPythonで評価する。
        state[6:8] += dt * alpha
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`state[4:6] +` を後続計算で使う明示的な中間量として設定する。 数式: `state[4:6] += dt * state[6:8]` の演算・変換をPythonで評価する。
        state[4:6] += dt * state[6:8]
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `logs.append(state.copy())` の要素または終端を対応付ける。
        logs.append(state.copy())

    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`h` を後続計算で使う明示的な中間量として設定する。 数式: `h = np.asarray(logs)` の演算・変換をPythonで評価する。
    h = np.asarray(logs)
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`z_rmse` を後続計算で使う明示的な中間量として設定する。 数式: `z_rmse = float(np.sqrt(np.mean((h[:,0]-0.30)**2)))` の演算・変換をPythonで評価する。
    z_rmse = float(np.sqrt(np.mean((h[:,0]-0.30)**2)))
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`vx_rmse` を後続計算で使う明示的な中間量として設定する。 数式: `vx_rmse = float(np.sqrt(np.mean((h[:,1]-cfg["vx_ref"])**2)))` の演算・変換をPythonで評価する。
    vx_rmse = float(np.sqrt(np.mean((h[:,1]-cfg["vx_ref"])**2)))
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`angle_rmse` を後続計算で使う明示的な中間量として設定する。 数式: `angle_rmse = float(np.sqrt(np.mean(h[:,4]**2 + (h[:,5]-slope)**2)))` の演算・変換をPythonで評価する。
    angle_rmse = float(np.sqrt(np.mean(h[:,4]**2 + (h[:,5]-slope)**2)))
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`fallen` を後続計算で使う明示的な中間量として設定する。 数式: `fallen = bool(np.any(h[:,0] < 0.16) or np.any(np.abs(h[:,4:6]) > 0.75))` の演算・変換をPythonで評価する。
    fallen = bool(np.any(h[:,0] < 0.16) or np.any(np.abs(h[:,4:6]) > 0.75))
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`mean_residual` を後続計算で使う明示的な中間量として設定する。 数式: `mean_residual = float(np.mean(residuals))` の演算・変換をPythonで評価する。
    mean_residual = float(np.mean(residuals))
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`sat_rate` を後続計算で使う明示的な中間量として設定する。 数式: `sat_rate = float(np.mean(tau_sats))` の演算・変換をPythonで評価する。
    sat_rate = float(np.mean(tau_sats))
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`solve_ms` を後続計算で使う明示的な中間量として設定する。 数式: `solve_ms = np.asarray(controller_times) * 1e3` の演算・変換をPythonで評価する。
    solve_ms = np.asarray(controller_times) * 1e3
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`score` を後続計算で使う明示的な中間量として設定する。 数式: `score = np.clip(100 - 180*z_rmse - 35*vx_rmse - 55*angle_rmse` の演算・変換をPythonで評価する。
    score = np.clip(100 - 180*z_rmse - 35*vx_rmse - 55*angle_rmse
                    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `- 45*mean_residual - 100*sat_rate - 100*fallen, 0, 100)` の要素または終端を対応付ける。 数式: `- 45*mean_residual - 100*sat_rate - 100*fallen, 0, 100)` の演算・変換をPythonで評価する。
                    - 45*mean_residual - 100*sat_rate - 100*fallen, 0, 100)
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`passed` を後続計算で使う明示的な中間量として設定する。 数式: `passed = (not fallen and z_rmse < 0.10 and vx_rmse < 0.40` の演算・変換をPythonで評価する。
    passed = (not fallen and z_rmse < 0.10 and vx_rmse < 0.40
              # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `and angle_rmse < 0.30 and mean_residual < 0.55 and sat_rate < 0.25)` の要素または終端を対応付ける。
              and angle_rmse < 0.30 and mean_residual < 0.55 and sat_rate < 0.25)
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`return {` の値を次の制御境界へ返す。 数式: `return {` の演算・変換をPythonで評価する。
    return {
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `"name": cfg["name"], "level": cfg["level"], "pass": bool(passed),` の要素または終端を対応付ける。
        "name": cfg["name"], "level": cfg["level"], "pass": bool(passed),
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `"score": float(score), "z_rmse_m": z_rmse, "vx_rmse_mps": vx_rmse,` の要素または終端を対応付ける。
        "score": float(score), "z_rmse_m": z_rmse, "vx_rmse_mps": vx_rmse,
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `"angle_rmse_rad": angle_rmse, "wrench_residual": mean_residual,` の要素または終端を対応付ける。
        "angle_rmse_rad": angle_rmse, "wrench_residual": mean_residual,
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `"min_friction_margin_N": float(np.min(margins)),` の要素または終端を対応付ける。 数式: `"min_friction_margin_N": float(np.min(margins)),` の演算・変換をPythonで評価する。
        "min_friction_margin_N": float(np.min(margins)),
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `"torque_sat_rate": sat_rate, "fallen": fallen,` の要素または終端を対応付ける。
        "torque_sat_rate": sat_rate, "fallen": fallen,
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `"controller_mean_ms": float(np.mean(solve_ms)),` の要素または終端を対応付ける。 数式: `"controller_mean_ms": float(np.mean(solve_ms)),` の演算・変換をPythonで評価する。
        "controller_mean_ms": float(np.mean(solve_ms)),
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `"controller_p95_ms": float(np.percentile(solve_ms, 95)),` の要素または終端を対応付ける。 数式: `"controller_p95_ms": float(np.percentile(solve_ms, 95)),` の演算・変換をPythonで評価する。
        "controller_p95_ms": float(np.percentile(solve_ms, 95)),
        # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `"realtime_margin_at_100Hz": float(10.0 / max(np.percentile(solve_ms,95),…` の要素または終端を対応付ける。 数式: `"realtime_margin_at_100Hz": float(10.0 / max(np.percentile(solve_ms,95),…` の演算・変換をPythonで評価する。
        "realtime_margin_at_100Hz": float(10.0 / max(np.percentile(solve_ms,95), 1e-9)),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `}` の要素または終端を対応付ける。
    }


In [ ]:
# --- Block G: 30 scenariosを実行して保存 ---
# 意図: 同じseed・同じmetricで難易度間を比較し、都合の良いrunだけを選ばない。
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`started` を後続計算で使う明示的な中間量として設定する。 数式: `started = time.perf_counter()` の演算・変換をPythonで評価する。
started = time.perf_counter()
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`results` を後続計算で使う明示的な中間量として設定する。 数式: `results = [run_scenario(cfg, seed=1000+i) for i, cfg in enumerate(SCENAR…` の演算・変換をPythonで評価する。
results = [run_scenario(cfg, seed=1000+i) for i, cfg in enumerate(SCENARIOS)]
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`elapsed` を後続計算で使う明示的な中間量として設定する。 数式: `elapsed = time.perf_counter() - started` の演算・変換をPythonで評価する。
elapsed = time.perf_counter() - started
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`df` を後続計算で使う明示的な中間量として設定する。 数式: `df = pd.DataFrame(results)` の演算・変換をPythonで評価する。
df = pd.DataFrame(results)

# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`csv_path` を後続計算で使う明示的な中間量として設定する。 数式: `csv_path = OUT / "legged_model_benchmark_30.csv"` の演算・変換をPythonで評価する。
csv_path = OUT / "legged_model_benchmark_30.csv"
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`json_path` を後続計算で使う明示的な中間量として設定する。 数式: `json_path = OUT / "legged_model_benchmark_30.json"` の演算・変換をPythonで評価する。
json_path = OUT / "legged_model_benchmark_30.json"
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`df.to_csv(csv_path, index` を後続計算で使う明示的な中間量として設定する。 数式: `df.to_csv(csv_path, index=False)` の演算・変換をPythonで評価する。
df.to_csv(csv_path, index=False)
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`json_path.write_text(json.dumps(results, ensure_ascii` を後続計算で使う明示的な中間量として設定する。 数式: `json_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), …` の演算・変換をPythonで評価する。
json_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`print(f"30 scenarios wall time: {elapsed:.3f} s")` の観測値を表示して判定根拠を残す。
print(f"30 scenarios wall time: {elapsed:.3f} s")
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`display(df)` の観測値を表示して判定根拠を残す。
display(df)
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`display(df.groupby("level").agg(` の観測値を表示して判定根拠を残す。
display(df.groupby("level").agg(
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`scenarios` を後続計算で使う明示的な中間量として設定する。 数式: `scenarios=("name","count"),` の演算・変換をPythonで評価する。
    scenarios=("name","count"),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`passed` を後続計算で使う明示的な中間量として設定する。 数式: `passed=("pass","sum"),` の演算・変換をPythonで評価する。
    passed=("pass","sum"),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`mean_score` を後続計算で使う明示的な中間量として設定する。 数式: `mean_score=("score","mean"),` の演算・変換をPythonで評価する。
    mean_score=("score","mean"),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`mean_vx_rmse` を後続計算で使う明示的な中間量として設定する。 数式: `mean_vx_rmse=("vx_rmse_mps","mean"),` の演算・変換をPythonで評価する。
    mean_vx_rmse=("vx_rmse_mps","mean"),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`mean_wrench_residual` を後続計算で使う明示的な中間量として設定する。 数式: `mean_wrench_residual=("wrench_residual","mean"),` の演算・変換をPythonで評価する。
    mean_wrench_residual=("wrench_residual","mean"),
    # 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`p95_controller_ms` を後続計算で使う明示的な中間量として設定する。 数式: `p95_controller_ms=("controller_p95_ms","max"),` の演算・変換をPythonで評価する。
    p95_controller_ms=("controller_p95_ms","max"),
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `))` の要素または終端を対応付ける。
))


In [ ]:
# --- Block H: 性能図 ---
# 上: score。下: 主要制約metric。色は難易度、×はfail。
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`colors` を後続計算で使う明示的な中間量として設定する。 数式: `colors = {"easy":"tab:green", "normal":"tab:blue", "hard":"tab:red"}` の演算・変換をPythonで評価する。
colors = {"easy":"tab:green", "normal":"tab:blue", "hard":"tab:red"}
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`fig, axes` を後続計算で使う明示的な中間量として設定する。 数式: `fig, axes = plt.subplots(2, 1, figsize=(14, 9), constrained_layout=True)` の演算・変換をPythonで評価する。
fig, axes = plt.subplots(2, 1, figsize=(14, 9), constrained_layout=True)
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`x` を後続計算で使う明示的な中間量として設定する。 数式: `x = np.arange(len(df))` の演算・変換をPythonで評価する。
x = np.arange(len(df))
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`bar_colors` を後続計算で使う明示的な中間量として設定する。 数式: `bar_colors = [colors[v] for v in df["level"]]` の演算・変換をPythonで評価する。
bar_colors = [colors[v] for v in df["level"]]
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`axes[0].bar(x, df["score"], color` を後続計算で使う明示的な中間量として設定する。 数式: `axes[0].bar(x, df["score"], color=bar_colors, alpha=0.85)` の演算・変換をPythonで評価する。
axes[0].bar(x, df["score"], color=bar_colors, alpha=0.85)
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`axes[0].scatter(x[~df["pass"]], df.loc[~df["pass"],"score"], marker` を後続計算で使う明示的な中間量として設定する。 数式: `axes[0].scatter(x[~df["pass"]], df.loc[~df["pass"],"score"], marker="x",…` の演算・変換をPythonで評価する。
axes[0].scatter(x[~df["pass"]], df.loc[~df["pass"],"score"], marker="x", s=80, color="black", label="fail")
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `axes[0].set_ylabel("model-level score [0-100]")` の要素または終端を対応付ける。 数式: `axes[0].set_ylabel("model-level score [0-100]")` の演算・変換をPythonで評価する。
axes[0].set_ylabel("model-level score [0-100]")
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `axes[0].set_title("30 scenario model benchmark — not end-to-end ROS/Gaze…` の要素または終端を対応付ける。 数式: `axes[0].set_title("30 scenario model benchmark — not end-to-end ROS/Gaze…` の演算・変換をPythonで評価する。
axes[0].set_title("30 scenario model benchmark — not end-to-end ROS/Gazebo performance")
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `axes[0].legend()` の要素または終端を対応付ける。
axes[0].legend()

# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`axes[1].plot(x, df["vx_rmse_mps"], "o-", label` を後続計算で使う明示的な中間量として設定する。 数式: `axes[1].plot(x, df["vx_rmse_mps"], "o-", label="vx RMSE [m/s]")` の演算・変換をPythonで評価する。
axes[1].plot(x, df["vx_rmse_mps"], "o-", label="vx RMSE [m/s]")
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`axes[1].plot(x, df["wrench_residual"], "s-", label` を後続計算で使う明示的な中間量として設定する。 数式: `axes[1].plot(x, df["wrench_residual"], "s-", label="normalized wrench re…` の演算・変換をPythonで評価する。
axes[1].plot(x, df["wrench_residual"], "s-", label="normalized wrench residual")
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`axes[1].plot(x, df["torque_sat_rate"], "^-", label` を後続計算で使う明示的な中間量として設定する。 数式: `axes[1].plot(x, df["torque_sat_rate"], "^-", label="torque saturation ra…` の演算・変換をPythonで評価する。
axes[1].plot(x, df["torque_sat_rate"], "^-", label="torque saturation rate")
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `axes[1].set_ylabel("metric")` の要素または終端を対応付ける。
axes[1].set_ylabel("metric")
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`axes[1].legend(ncol` を後続計算で使う明示的な中間量として設定する。 数式: `axes[1].legend(ncol=3)` の演算・変換をPythonで評価する。
axes[1].legend(ncol=3)
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`axes[1].set_xticks(x, df["name"], rotation` を後続計算で使う明示的な中間量として設定する。 数式: `axes[1].set_xticks(x, df["name"], rotation=75, ha="right")` の演算・変換をPythonで評価する。
axes[1].set_xticks(x, df["name"], rotation=75, ha="right")
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`fig_path` を後続計算で使う明示的な中間量として設定する。 数式: `fig_path = OUT / "legged_model_benchmark_30.png"` の演算・変換をPythonで評価する。
fig_path = OUT / "legged_model_benchmark_30.png"
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`fig.savefig(fig_path, dpi` を後続計算で使う明示的な中間量として設定する。 数式: `fig.savefig(fig_path, dpi=150)` の演算・変換をPythonで評価する。
fig.savefig(fig_path, dpi=150)
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、直前の式・構造へ `plt.show()` の要素または終端を対応付ける。
plt.show()
# 背景: 教育用proxyは上流stackの性能試験ではない。目的: 縮約model内だけの制約・追従指標を比較するため、`print(fig_path)` の観測値を表示して判定根拠を残す。
print(fig_path)


## 結果の分析方法
次の順で失敗原因を読む。

1. `fallen`: 縮約plantが高さ/姿勢限界を超えた。
2. `wrench_residual`: 接触幾何と摩擦で要求wrenchを作れない。
3. `min_friction_margin_N`: 0付近は摩擦境界がactive。
4. `torque_sat_rate`: forceは作れても関節torque proxyで飽和。
5. `controller_p95_ms`: 100 Hzの10 ms deadlineに対する計算余裕。

## この先の本当のrepository benchmark
同じ30 scenario schemaを次のROS/Gazebo項目へ移植する。

- `task.info`, `reference.info`, `gait.info` をscenarioごとに生成
- `/cmd_vel` とgait commandをtimestamp付きpublish
- `/legged_robot_mpc_observation`, odometry, joint state, WBC/solver統計をrosbag保存
- 同じRMSE、転倒、constraint、torque、deadline指標を計算
- A1 URDF、Gazebo contact、OCS2/qpOASESを含む結果だけを上流end-to-end性能と呼ぶ

現環境ではROS/OCS2が無いため、そこを実行済みと偽らない。
実際のA1 MuJoCo plant、20秒以上のGIF、物理閾値を使う実行証拠は
`14_a1_mujoco_benchmark_30_scenarios.ipynb` へ進む。
